In [1]:
import os
os.chdir('application')

In [2]:

# from typing import List
# from pydantic import BaseModel
import redis

from modules.redis import RedisConversationMemory
# from modules.retriever_wrappper import SanitizingRetriever
from modules.history_utils import build_chat_history_messages

from config import (REDIS_URL, MAX_HISTORY_MESSAGES, 
                    gemini_chat_model, api_key_name, system_prompt, qa_system_prompt
)
from vectorstore.milvus_vectorstore import get_vectorstore

# import logging

# LangChain imports (use provider wrapper)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.output_parsers import PydanticOutputParser

C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from schemas.rag_outputs import AnswerWithSources

In [4]:
# from langchain_core.runnables import RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser
# from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

In [5]:
from modules.utils import format_docs_with_metadata

In [6]:
# ----- CONFIG -----
r = redis.Redis.from_url(REDIS_URL, decode_responses=True)
# app = FastAPI()

In [7]:
# Your already-created vectorstore (LangChain Milvus wrapper)
vectorstore = get_vectorstore()

In [8]:
mem = RedisConversationMemory(session_id='001', max_messages=MAX_HISTORY_MESSAGES)


In [9]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

In [10]:
llm = ChatGoogleGenerativeAI(model=gemini_chat_model, google_api_key=os.environ.get(api_key_name))

In [11]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt+'ini adalah sesi debugging. Tambahkan retrieval text pada field "answer"'),
    MessagesPlaceholder(variable_name="chat_history"),            # inject conversation history here
#     ("system", "Gunakan dokumen yang diekstrak berikut untuk menjawab pertanyaan user:\n\n{context}"),
    ("human", "{question}")
])

In [12]:
pydantic_parser = PydanticOutputParser(pydantic_object=AnswerWithSources)

In [13]:
rag_chain = (
    {
        # The retriever runs first, using the original question.
        # Its output is passed to format_docs and becomes the 'context'.
        "context": (lambda x: x["question"]) | retriever | format_docs_with_metadata,
         
        # The original 'question' is passed through unchanged.
        "question": (lambda x: x["question"]),
        
        # The 'chat_history' is passed through unchanged.
        "chat_history": (lambda x: x["chat_history"])
    }
    | qa_prompt
    | llm
    | pydantic_parser
)

In [14]:
raw_history = mem.get_chat_history_pairs() 
raw_history

[]

In [15]:
question = 'Halo, nama saya Henry. Salam kenal. Apa yang bisa anda bantu?'

In [16]:
chat_history_messages = build_chat_history_messages(raw_history)
chat_history_messages

[]

In [17]:
# The invoke call now passes a single dictionary with all the required keys
result = rag_chain.invoke({
    "question": question, 
    "chat_history": chat_history_messages
})
result

AnswerWithSources(answer='Halo, Bapak Henry. Salam kenal juga. Saya Nastari, asisten AI di bidang HR KPN-Corp. Saya di sini untuk membantu Bapak dengan pertanyaan terkait kebijakan perusahaan dan masalah karyawan. Silakan sampaikan apa yang ingin Bapak tanyakan.', sources=[])

In [22]:
# type(result)

schemas.rag_outputs.AnswerWithSources

In [19]:
# from modules.utils import extract_and_convert_dictionary_string

In [23]:
# result = extract_and_convert_dictionary_string(result)
# result

In [25]:
result.answer

'Halo, Bapak Henry. Salam kenal juga. Saya Nastari, asisten AI di bidang HR KPN-Corp. Saya di sini untuk membantu Bapak dengan pertanyaan terkait kebijakan perusahaan dan masalah karyawan. Silakan sampaikan apa yang ingin Bapak tanyakan.'

In [27]:
assistant_text = result.answer or result.result or ""
sources_meta = result.sources or []
# sources_meta = [d for d in source_docs]

assistant_text, sources_meta

('Halo, Bapak Henry. Salam kenal juga. Saya Nastari, asisten AI di bidang HR KPN-Corp. Saya di sini untuk membantu Bapak dengan pertanyaan terkait kebijakan perusahaan dan masalah karyawan. Silakan sampaikan apa yang ingin Bapak tanyakan.',
 [])

In [28]:
# 10) Save the turn into Redis after successful call: user then assistant
mem.add_user_message(question)
mem.add_assistant_message(assistant_text, sources=sources_meta)

In [33]:
raw_history = mem.get_chat_history_pairs() 
raw_history

[('Halo, nama saya Henry. Salam kenal. Apa yang bisa anda bantu?',
  'Halo, Bapak Henry. Salam kenal juga. Saya Nastari, asisten AI di bidang HR KPN-Corp. Saya di sini untuk membantu Bapak dengan pertanyaan terkait kebijakan perusahaan dan masalah karyawan. Silakan sampaikan apa yang ingin Bapak tanyakan.')]

In [34]:
chat_history_messages = build_chat_history_messages(raw_history)
chat_history_messages

[HumanMessage(content='Halo, nama saya Henry. Salam kenal. Apa yang bisa anda bantu?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Halo, Bapak Henry. Salam kenal juga. Saya Nastari, asisten AI di bidang HR KPN-Corp. Saya di sini untuk membantu Bapak dengan pertanyaan terkait kebijakan perusahaan dan masalah karyawan. Silakan sampaikan apa yang ingin Bapak tanyakan.', additional_kwargs={}, response_metadata={})]

In [35]:
question2 = "apakah anda mengenal saya?"

In [36]:
# The invoke call now passes a single dictionary with all the required keys
result = rag_chain.invoke({
    "question": question2, 
    "chat_history": chat_history_messages
})
result

AnswerWithSources(answer='Sebagai asisten AI, saya tidak memiliki kemampuan untuk mengenal individu secara pribadi, Bapak Henry. Saya di sini untuk membantu Bapak dengan informasi terkait kebijakan HR KPN-Corp berdasarkan dokumen yang saya miliki. Ada yang bisa saya bantu terkait hal tersebut?', sources=[])

In [ ]:
stop

In [ ]:
conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=None,  # ok if you pass chat_history manually
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": qa_prompt},
)

In [ ]:
result = conv_chain.invoke({"question": question, "chat_history": chat_history_messages})
result

In [ ]:
from langchain.chains import (
    create_history_aware_retriever,
)
from langchain.chains.combine_documents import create_stuff_documents_chain

In [ ]:
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
# contextualize_q_prompt

In [ ]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [ ]:
qa_prompt.input_variables

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)